## Human-In-The-Loop Workflows with LangChain

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-anthropic==1.5.4 anthropic==0.120.2 python-dotenv==1.2.2 langgraph==1.2.10

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

### Instantiating the ChatAnthropic Class

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key
)

### Create a Refund Tool

In [ ]:
@tool
def issue_refund(
    customer_name: str,
    amount: float
) -> str:
    """Issue a refund to a customer."""

    return (
        f"Refund of ${amount:.2f} "
        f"successfully issued to {customer_name}."
    )

### Add Human in the Loop

In [ ]:
agent = create_agent(
    model=model,

    tools=[
        issue_refund
    ],

    system_prompt=(
        "You are a customer service agent. "
        "Help customers with refund requests. "
        "Use the issue_refund tool when a refund "
        "needs to be processed."
    ),

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "issue_refund": {
                    "allowed_decisions": [
                        "approve",
                        "edit",
                        "reject"
                    ]
                }
            }
        )
    ],

    checkpointer=InMemorySaver(),
)

### Create a Conversation Thread

In [ ]:
config = {
    "configurable": {
        "thread_id": "refund-demo-1"
    }
}

### Ask the Agent to issue a Refund

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Customer John was accidentally "
                    "charged $75 twice. Please refund "
                    "one of the charges."
                )
            }
        ]
    },
    config=config,
    version="v2",
)

interrupt = result.interrupts[0].value

action = interrupt["action_requests"][0]

print("ACTION REQUIRES APPROVAL")
print("------------------------")
print("Tool:", action["name"])

### Human approves the Refund

In [ ]:
result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config,
    version="v2",
)

In [ ]:
for message in result.value["messages"]:

    if message.type == "ai" and message.content:
        print(message.content)